In [1]:
import pandas as pd
import re
import os

# Load the raw data
DATA_PATH = os.path.join("..", "data", "raw", "IMDB Dataset.csv")
df = pd.read_csv(DATA_PATH)

print(f"Loaded {len(df)} rows.")

Loaded 50000 rows.


In [2]:
# Find a review containing HTML break tags
dirty_reviews = df[df['review'].str.contains('<br />', case=False, na=False)]

# Save the index of the first dirty review for testing
test_idx = dirty_reviews.index[0]
raw_text = df.loc[test_idx, 'review']

print("--- RAW TEXT SNIPPET ---")
print(raw_text[:300]) # Print first 300 characters

--- RAW TEXT SNIPPET ---
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru


In [3]:
def standardize_text(text: str) -> str:
    """
    Converts text to lowercase, removes HTML tags, 
    and strips stray whitespace/line breaks.
    """
    if not isinstance(text, str):
        return ""
        
    # 1. Convert to lowercase
    text = text.lower()
    
    # 2. Remove HTML tags (matches anything between < and >)
    # Replacing with a space prevents word merging (e.g., "hello<br/>world" -> "hello world")
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3. Strip stray line breaks and tabs
    text = re.sub(r'[\n\t\r]', ' ', text)
    
    # 4. Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [4]:
# Verify on our test case
cleaned_test_text = standardize_text(raw_text)

print("--- CLEANED TEXT SNIPPET ---")
print(cleaned_test_text[:300])
print("\nSuccess: <br /> tags removed!" if "<br />" not in cleaned_test_text else "\nFailed: Tags still present.")

# Apply to the entire DataFrame
print("\nApplying standardization to the entire dataset (this may take a few seconds)...")
df['clean_review'] = df['review'].apply(standardize_text)

# Preview the new column
df[['review', 'clean_review']].head(3)

--- CLEANED TEXT SNIPPET ---
one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me. the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this

Success: <br /> tags removed!

Applying standardization to the entire dataset (this may take a few seconds)...


,review,clean_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production. the filming tec...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...


In [5]:
import nltk
from nltk.corpus import stopwords

# Download required NLTK data (runs once per environment)
nltk.download('stopwords')
nltk.download('wordnet')
# Downloading 'punkt' is also recommended for advanced tokenization
nltk.download('punkt')

# Load the English stopwords set (using a set makes lookups O(1) for speed)
stop_words = set(stopwords.words('english'))
print(f"Loaded {len(stop_words)} English stopwords.")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mysel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mysel\AppData\Roaming\nltk_data...


Loaded 198 English stopwords.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mysel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
def process_review(text: str) -> str:
    """
    Complete preprocessing pipeline:
    1. Lowercase & strip HTML
    2. Remove punctuation, special characters, and digits
    3. Remove stopwords
    4. Collapse whitespace
    """
    if not isinstance(text, str):
        return ""
        
    # 1. Lowercase and remove HTML (from previous step)
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 2. Remove punctuation, special characters, and digits
    # [^a-z\s] means "match anything that is NOT a lowercase letter or whitespace"
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 3. Tokenize and remove stopwords
    # .split() inherently handles collapsing multiple spaces and stripping ends
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    # 4. Join back into a single string 
    # (Scikit-learn's TF-IDF vectorizer expects strings, not lists of tokens)
    clean_text = " ".join(filtered_tokens)
    
    return clean_text

In [7]:
# Let's test on a noisy dummy string
test_string = "I LOVED this movie!!! <br /><br /> It was a 10/10, totally amazing... but the ending was bad."
print("--- TEST STRING BEFORE ---")
print(test_string)

print("\n--- TEST STRING AFTER ---")
print(process_review(test_string))
# Expected output: "loved movie totally amazing ending bad"

# Apply the full pipeline to the dataset (overwriting the previous clean column)
print("\nProcessing 50,000 reviews (this may take 10-20 seconds)...")
df['clean_review'] = df['review'].apply(process_review)

# Preview the final cleaned output vs raw
df[['review', 'clean_review']].head()

--- TEST STRING BEFORE ---
I LOVED this movie!!! <br /><br /> It was a 10/10, totally amazing... but the ending was bad.

--- TEST STRING AFTER ---
loved movie totally amazing ending bad

Processing 50,000 reviews (this may take 10-20 seconds)...


,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching oz episode ho...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake thinks zombie...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [8]:
from nltk.stem import WordNetLemmatizer

# Initialize WordNet Lemmatizer
lemmatizer = WordNetLemmatizer()

def process_and_lemmatize_review(text: str) -> str:
    """
    Full text preprocessing pipeline:
    1. Lowercase & strip HTML tags
    2. Remove non-alphabetic characters & digits
    3. Tokenize & filter English stopwords
    4. Lemmatize tokens to root form (e.g., 'movies' -> 'movie', 'running' -> 'running')
    5. Rejoin into a single space-separated string
    """
    if not isinstance(text, str):
        return ""
        
    # 1. Lowercase & remove HTML
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 2. Remove punctuation, special characters, and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 3. Tokenize (split on whitespace)
    tokens = text.split()
    
    # 4. Stopword filtering + Lemmatization
    lemmatized_tokens = [
        lemmatizer.lemmatize(word) 
        for word in tokens 
        if word not in stop_words
    ]
    
    # 5. Rejoin tokens into a single clean string
    return " ".join(lemmatized_tokens)

In [9]:
# Define output destination
PROCESSED_PATH = os.path.join("..", "data", "processed", "imdb_clean.csv")

# Create data/processed directory if missing
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)

# Save clean reviews and sentiment labels
df[['clean_review', 'sentiment']].to_csv(PROCESSED_PATH, index=False)
print(f"✅ Processed dataset saved to: {PROCESSED_PATH}")

✅ Processed dataset saved to: ..\data\processed\imdb_clean.csv


In [10]:
# Select 3 random rows to spot-check transformation quality
sample_df = df[['review', 'clean_review']].sample(3, random_state=42)

for idx, row in sample_df.iterrows():
    print(f"--- ROW {idx} ---")
    print(f"RAW   : {row['review'][:200]}...")
    print(f"CLEAN : {row['clean_review'][:200]}...\n")

--- ROW 33553 ---
RAW   : I really liked this Summerslam due to the look of the arena, the curtains and just the look overall was interesting to me for some reason. Anyways, this could have been one of the best Summerslam's ev...
CLEAN : really liked summerslam due look arena curtains look overall interesting reason anyways could one best summerslam ever wwf lex luger main event yokozuna time ok huge fat man vs strong man glad times c...

--- ROW 9427 ---
RAW   : Not many television shows appeal to quite as many different kinds of fans like Farscape does...I know youngsters and 30/40+ years old;fans both Male and Female in as many different countries as you ca...
CLEAN : many television shows appeal quite many different kinds fans like farscape know youngsters years old fans male female many different countries think adore v miniseries elements found almost every show...

--- ROW 199 ---
RAW   : The film quickly gets to a major chase scene with ever increasing destruction. The first re

In [11]:
# Check for nulls or blank strings after cleaning
null_count = df['clean_review'].isnull().sum()
empty_mask = df['clean_review'].str.strip() == ''
empty_count = empty_mask.sum()

print(f"Null cleaned reviews : {null_count}")
print(f"Empty cleaned reviews: {empty_count}")

# Inspect the raw text of reviews that turned out empty
if empty_count > 0:
    print("\n--- Inspecting raw reviews that became empty after cleaning ---")
    print(df[empty_mask][['review', 'sentiment']])

Null cleaned reviews : 0
Empty cleaned reviews: 0


In [13]:
initial_len = len(df)

# Drop empty or null cleaned reviews
df_clean = df[~empty_mask & ~df['clean_review'].isnull()].copy()
df_clean.reset_index(drop=True, inplace=True)

dropped_count = initial_len - len(df_clean)
print(f"Dropped {dropped_count} empty row(s). Final dataset size: {len(df_clean):,} rows.")

Dropped 0 empty row(s). Final dataset size: 50,000 rows.


In [14]:
import os

PROCESSED_PATH = os.path.join("..", "data", "processed", "imdb_clean.csv")
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)

# Save both original and cleaned reviews along with sentiment labels
df_clean[['review', 'clean_review', 'sentiment']].to_csv(PROCESSED_PATH, index=False)

print(f"✅ Verified cleaned dataset saved to: {PROCESSED_PATH}")

✅ Verified cleaned dataset saved to: ..\data\processed\imdb_clean.csv
